In [ ]:
import gc
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, convolve
import scipy.io as sio

#Loading raw EEG file

raw = mne.io.read_raw_egi('SUB2888_ses03_20241216_034412.mff', preload=False) #If uploading BIDS-formatted EEG data, mne.io.read_raw_brainvision works as well

sfreq = raw.info['sfreq']
picks = mne.pick_types(raw.info, eeg=True, exclude=[])[:128]
data = np.empty((len(picks), raw.n_times), dtype=np.float32)
for i, p in enumerate(picks):
    data[i] = raw.get_data(picks=[p])

print(f'sfreq={sfreq} Hz, channels={data.shape[0]}, samples={data.shape[1]}, dtype={data.dtype}')

In [ ]:
#Defining OMA parameters

M = 1030 #TREV spacing is 1030 samples
J = 20000 #can experiment with larger or smaller J but 20,000 seemed to be optimal
L = 1 #since TR rate is stable, L=1

In [ ]:
#OMA filter implementation

def h_oma_frequency_response(n_fft, M):
    """Zero-phase OMA magnitude on DFT bins: |sin(M ω/2)/sin(ω/2)|^2 / M^2
    (HOMA evaluated on the unit circle)"""
    k = np.arange(n_fft)
    omega = 2.0 * np.pi * k / n_fft
    num = np.sin(M * omega / 2.0)
    den = np.sin(omega / 2.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(np.abs(den) < 1e-15, M, num / den)
    h = (ratio / M) ** 2
    return np.clip(h.real, 0.0, None)


def h_l_frequency_response(n_fft, M, J, L):
    """Final cascaded OMA transfer function H_L on DFT bins
    This whole function is unnecessary if there is no significant time jitter"""
    h_oma = h_oma_frequency_response(n_fft, M)
    one_minus = np.clip(1.0 - h_oma, 0.0, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        log_om = np.full_like(one_minus, -np.inf)
        pos = one_minus > 0
        log_om[pos] = J * np.log(one_minus[pos])
        powered = np.exp(log_om)
        powered[~np.isfinite(powered)] = 0.0
    h_c = 1.0 - powered
    return np.clip(h_c, 0.0, 1.0) ** L

def oma_filter_zdomain(s, M, J=200_000, L=1, pad=True, inplace=False):
    """
    s : array of(n_channels, n_times)
    M : TR-slice window order in samples, 1030 for all subjects
    J : Number of iterative-filter iterations
    L : Number of cascades. if no significant time jitter, L=1
    pad : bool, zero-pad by ~2M samples to reduce circular wrap-around
    inplace : bool, If True overwrite s channel-by-channel to save memory
    """
    s = np.asarray(s, dtype=np.float32)
    single = s.ndim == 1
    if single:
        s = s[np.newaxis, :]
        inplace = False
    n_ch, n_times = s.shape
    pad_len = int(np.ceil(2 * M)) if pad else 0
    n_fft = n_times + 2 * pad_len
    h_l = h_l_frequency_response(n_fft, M, J, L)[: n_fft // 2 + 1].astype(np.float32, copy=False) #similar results with float32 and float64 but the latter was making my computer cras

    out = s if inplace else np.empty_like(s)
    for ch in range(n_ch):
        x = np.pad(s[ch], pad_len, mode='constant') if pad_len else s[ch].astype(np.float32, copy=True)
        X = np.fft.rfft(x, n=n_fft)
        y = np.fft.irfft(X * h_l, n=n_fft)
        out[ch] = y[pad_len:pad_len + n_times] if pad_len else y[:n_times]
        if (ch + 1) % 16 == 0 or ch == n_ch - 1:
            print(f'  OMA channel {ch + 1}/{n_ch}', flush=True)
    return out[0] if single else out

def center_eeg_array(x):
    """Average-reference then per-channel demean (matches AAS baseline for plotting)."""
    x = np.asarray(x, dtype=np.float32)
    x -= x.mean(axis=0, keepdims=True)
    x -= x.mean(axis=1, keepdims=True)
    return x

#overwriting data in place for memory efficiency
result_z = oma_filter_zdomain(data, M=M, J=J, L=L, inplace=True)
result_z = center_eeg_array(result_z)
data = result_z
print('OMA done:', result_z.shape, result_z.dtype)
print(f'post-center mean ({eeg_labels[ch_idx]}): {result_z[ch_idx].mean():.3e}')

In [ ]:
#Saves the cleaned signal as a .mat file

sio.savemat(
    'oma_cleaned_signal.mat',
    {
        'cleaned_eeg_data': result_z.astype(np.float32, copy=False),
        'srate': np.array([[sfreq]]),
        'chanlabels': np.array(eeg_labels, dtype=object),
        'M': np.array([[M]]),
        'J': np.array([[J]]),
        'L': np.array([[L]]),
        'method': np.array(['oma_zdomain'], dtype=object),
    },
    do_compression=True,
)
print('Saved oma_cleaned_signal.mat (float32, compressed)')
